In [120]:
# conda activate genomic_tools

library(ensembldb)
library(data.table)
library(GenomicRanges)
library(EnsDb.Hsapiens.v86)

In [121]:
# Compile list of all exons from cell type splicing analysis
exon_lookup <- list()
for (file in list.files("data/ctype_exons")) {
    if (grepl("csv$", file)) {
        signif_exons_df <- fread(paste0("data/ctype_exons/", file), data.table=FALSE)
        signif_exons_df <- signif_exons_df[!is.na(signif_exons_df[['Oligo']]), ]
        for (i in 1:nrow(signif_exons_df)) {
            row <- signif_exons_df[i, ]
            event <- as.character(row[1])
            if (event %in% names(exon_lookup)) {
                next
            }
            exon_lookup[[event]] <- data.frame(
                    gene=row['Gene'],
                    chr=row['chr'],
                    strand=row['strand'],
                    exon_start=row['exon_start'],
                    exon_end=row['exon_end']
                )
        } 
    }
}

In [172]:
library(AnnotationHub)
ah <- AnnotationHub()
query(ah, c("EnsDb", "Hsapiens", "108"))

ERROR: Error in library(AnnotationHub): there is no package called 'AnnotationHub'


In [122]:
exon_df <- do.call(rbind, exon_lookup)

In [124]:
exon_df_subset <- exon_df[1:10, ]

In [125]:
exon_gr <- GRanges(
    seqnames=sapply(strsplit(x = exon_df_subset$chr, split="chr"), function(x) x[2]),
    ranges=IRanges(start=exon_df_subset$exon_start, end=exon_df_subset$exon_end),
    names=rownames(exon_df_subset),
    strand=strand(exon_df_subset$strand)
)

In [128]:
db <- EnsDb.Hsapiens.v86
db_filtered <- filter(db, filter=TxBiotypeFilter("protein_coding"))

In [ ]:
g2p <- genomeToProtein(exon_gr_clipped, db_filtered)
g2p_df <- as.data.frame(g2p)
head(g2p_df)

Warning message:
"Provided coordinates for 'ENST00000314070', 'ENST00000462234', 'ENST00000376460', 'ENST00000629219' are not within the coding region"
Warning message:
"The CDS of 'ENST00000446759', 'ENST00000339554' does not match the length of the encoded protein. Returned protein coordinates for this/these transcript(s) might not be correct"


group,group_name,start,end,width,names,tx_id,cds_ok,exon_id,exon_rank,seq_start,seq_end,seq_name,seq_strand
<int>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<lgl>,<chr>,<int>,<int>,<int>,<chr>,<chr>
1,NA,15,65,51,ENSP00000403049,ENST00000446759,FALSE,ENSE00003461009,3,77743016,77743165,13,+
2,NA,-1,-1,1,,ENST00000376460,NA,ENSE00001607866,56,98796206,98796213,13,-
3,NA,1790,1791,2,ENSP00000365643,ENST00000376460,TRUE,ENSE00001742044,49,98809123,98809128,13,-
3,NA,1790,1791,2,ENSP00000406883,ENST00000442173,TRUE,ENSE00001742044,49,98809123,98809128,13,-
4,NA,276,284,9,ENSP00000289290,ENST00000289290,TRUE,ENSE00001947677,10,115640110,115640136,X,+
4,NA,276,284,9,ENSP00000398945,ENST00000420625,TRUE,ENSE00001947677,8,115640110,115640136,X,+
5,NA,143,145,3,ENSP00000444957,ENST00000545766,TRUE,ENSE00003743980,5,15828192,15828200,X,-
6,NA,254,263,10,ENSP00000378338,ENST00000394869,TRUE,ENSE00003498768,8,29578962,29578988,17,-
6,NA,254,263,10,ENSP00000462775,ENST00000581348,TRUE,ENSE00003498768,8,29578962,29578988,17,-


In [ ]:
colnames(g2p_df)[ncol(g2p_df)] <- "event"  
exon_df$event = rownames(exon_df)

In [155]:
merge(g2p_df, exon_df, by.x="event", by.y="event", all.x=TRUE)

event,group,group_name,start,end,width,names.x,tx_id,cds_ok,exon_id,exon_rank,seq_start,seq_end,seq_name,seq_strand,Gene,chr,strand,exon_start,exon_end,names.y
<chr>,<int>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<lgl>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<chr>
ENSG00000088387_ProteinCoding_1,1,NA,-1,-1,1,,ENST00000376460,NA,ENSE00001607866,56,98796206,98796213,13,-,DOCK9,chr13,-,98796206,98796213,ENSG00000088387_ProteinCoding_1
ENSG00000088387_ProteinCoding_3,3,NA,1790,1791,2,ENSP00000365643,ENST00000376460,TRUE,ENSE00001742044,49,98809123,98809128,13,-,DOCK9,chr13,-,98809123,98809128,ENSG00000088387_ProteinCoding_3
ENSG00000088387_ProteinCoding_3,3,NA,1790,1791,2,ENSP00000406883,ENST00000442173,TRUE,ENSE00001742044,49,98809123,98809128,13,-,DOCK9,chr13,-,98809123,98809128,ENSG00000088387_ProteinCoding_3
ENSG00000102024_ProteinCoding_2,8,NA,276,284,9,ENSP00000289290,ENST00000289290,TRUE,ENSE00001947677,10,115640110,115640136,X,+,PLS3,chrX,+,115640110,115640136,ENSG00000102024_ProteinCoding_2
ENSG00000102024_ProteinCoding_2,8,NA,276,284,9,ENSP00000398945,ENST00000420625,TRUE,ENSE00001947677,8,115640110,115640136,X,+,PLS3,chrX,+,115640110,115640136,ENSG00000102024_ProteinCoding_2
ENSG00000108262_ProteinCoding_1,4,NA,254,263,10,ENSP00000378338,ENST00000394869,TRUE,ENSE00003498768,8,29578962,29578988,17,-,GIT1,chr17,-,29578962,29578988,ENSG00000108262_ProteinCoding_1
ENSG00000108262_ProteinCoding_1,4,NA,254,263,10,ENSP00000462775,ENST00000581348,TRUE,ENSE00003498768,8,29578962,29578988,17,-,GIT1,chr17,-,29578962,29578988,ENSG00000108262_ProteinCoding_1
ENSG00000139737_ProteinCoding_1,9,NA,15,65,51,ENSP00000403049,ENST00000446759,FALSE,ENSE00003461009,3,77743016,77743165,13,+,SLAIN1,chr13,+,77743016,77743165,ENSG00000139737_ProteinCoding_1
ENSG00000144283_NMD_2,7,NA,-1,-1,1,,,NA,NA,NA,158533438,158533534,2,+,PKP4,chr2,+,158533438,158533534,ENSG00000144283_NMD_2


In [ ]:
# Filter to valid mappings for this exon
valid <- g2p_df[g2p_df$start > 0 & !is.na(g2p_df$names), ]

# Get protein sequences
prot_seqs <- proteins(db_filtered,
                      filter=ProteinIdFilter(unique(valid$names)),
                      return.type="AAStringSet")

# Extract subsequences
valid$aa_seq <- mapply(function(prot_id, start, end) {
    as.character(subseq(prot_seqs[prot_id], start=start, end=end))
}, valid$names, valid$start, valid$end)